# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup: connect to the warehouse (token stays private)

This notebook reads one month of the real warehouse release straight from Hugging Face with DuckDB. Nothing is downloaded in full. **Never paste your token into a code cell**: the repo is public. Use the Colab Secrets panel (key named `HF_TOKEN`) or the private prompt below.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass

# Token order: environment variable -> Colab Secret -> private prompt (last resort).
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

In [ ]:
import duckdb
import numpy as np
import pandas as pd

con = duckdb.connect()
try:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
except Exception as e:
    # never let the token leak into a saved notebook through an error message
    raise RuntimeError(str(e).replace(HF_TOKEN, "***")) from None

REL   = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"                       # a mid-panel month, NOT the sealed final month
DAILY = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

DECISION_DAY = "2026-03-15"             # the moment I score pages: end of 15 March
EARLY_START, EARLY_END = "2026-03-01", "2026-03-15"   # feature window (15 days)
LATE_START,  LATE_END  = "2026-03-16", "2026-03-31"   # outcome window (16 days)
print("Connected. Slice:", MONTH, "| decision day:", DECISION_DAY)

## 1. Unit of analysis + time window

**Lane:** Refresh / Content Opportunity Scoring, now on the full warehouse instead of the 30,000-row starter file. These are my five plain-words contract answers.

1. **What one row means.** In the raw table, one row is **one page, for one client, on one day**. In my modelling frame, one row is **one page**, summarised over the first half of March 2026.
2. **Which table I use.** Only `fact_content_daily_performance`, and only its `month=2026-03` partition. March is a middle month of the panel, so I am not developing inside the final month that is kept as the outcome window for later work.
3. **Which time window.** My decision moment is the **end of 15 March 2026**. Every feature uses only 1 to 15 March, because those days are already in the past at that moment. The outcome I check afterwards is 16 to 31 March.
4. **What I predict (a proxy).** Whether a page's average daily search impressions in 16 to 31 March are **more than 20% lower** than in 1 to 15 March. This is a measurable stand-in for "this page is losing visibility", not proof that the page needs a refresh. To avoid noise from tiny pages I only keep pages with at least 100 impressions in 1 to 15 March.
5. **One thing I deliberately exclude.** **Everything dated 16 March or later is excluded from the features**, because it contains the answer. The outcome window is used only to build the label.

In [ ]:
contract = pd.DataFrame([
    ("1. One row means",       "raw: one page x one client x one day | model frame: one page, 1-15 Mar 2026"),
    ("2. Table used",          "fact_content_daily_performance, partition month=2026-03"),
    ("3. Time window",         f"features {EARLY_START}..{EARLY_END} (decision day {DECISION_DAY}); outcome {LATE_START}..{LATE_END}"),
    ("4. Label (proxy)",       "avg daily impressions in the outcome window are >20% lower than in the feature window"),
    ("5. Deliberately excluded", f"all data dated after {DECISION_DAY} (it contains the answer)"),
], columns=["contract item", "answer"])
contract.style.hide(axis="index").set_properties(**{"text-align": "left"})

## 2. Fields: feature / label / context / excluded

Every field I plan to touch goes in exactly one bucket. Excluded fields get a one-line reason.

- **Features (five at most, all from 1 to 15 March):** early impressions, early click-through rate, early average position, number of active days, and early momentum. The reason each one is knowable on 15 March is written next to it in section 3.
- **Label:** `declining`, built from impressions in 16 to 31 March compared with 1 to 15 March.
- **Context (grouping and joining only, never learned from):** `client_hash_id`, `content_hash_id`, `report_date`. I use the client id to keep a whole client on one side of the train/test split.
- **Excluded, with a why:**
  - all columns dated 16 March or later: they contain the answer;
  - the GA4 and session columns (`ga4_*`, `sessions_*`, `ai_*`, `scroll_events`): GA4 history starts at different dates per client and the flag can be missing, and this lane is about search impressions, so I do not need them;
  - the `*_data_available` flags themselves are used only to filter rows (with `IS TRUE`), not as features.

In [ ]:
fields = pd.DataFrame([
    ("imp_early",           "feature",  "sum of gsc_impressions, 1-15 Mar"),
    ("ctr_early_pct",       "feature",  "gsc_clicks / gsc_impressions x 100, 1-15 Mar"),
    ("pos_early",           "feature",  "mean of gsc_avg_position on days with impressions, 1-15 Mar"),
    ("active_days_early",   "feature",  "days with at least 1 impression, 1-15 Mar"),
    ("early_trend",         "feature",  "avg daily impressions 8-15 Mar divided by avg daily impressions 1-7 Mar"),
    ("declining",           "label",    "late avg daily impressions < 0.8 x early avg daily impressions"),
    ("client_hash_id",      "context",  "pseudonymous client; used to split train/test by client"),
    ("content_hash_id",     "context",  "pseudonymous page; grouping only"),
    ("data from 16 Mar on", "excluded", "future information; it contains the answer"),
    ("ga4_*, sessions_*, ai_*, scroll_events", "excluded", "GA4 starts at different dates and flags can be NULL; not needed for a search-impressions lane"),
], columns=["field", "bucket", "definition / why"])
fields.style.hide(axis="index").set_properties(**{"text-align": "left"})

## 3. Verify it with queries

A contract claim without a query next to it is a guess. These are my **three verification queries**, each run on the March partition. Their outputs are shown under each cell, and the sentence above each one says what the output should prove.

**Verification query 1: the grain.** I said one row is one page on one day for one client. If that is true, no (day, client, page) combination should appear twice, so the number of duplicated keys should be **0**.

In [ ]:
q1 = con.sql(f"""
    SELECT COUNT(*) AS duplicated_keys
    FROM (
        SELECT report_date, client_hash_id, content_hash_id
        FROM {DAILY}
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
""").df()
q1

**Verification query 2: row count and date span.** This tells me how big my slice is and whether it really covers only March 2026. The first and last day should be 1 and 31 March, and the number of distinct days should be 31.

In [ ]:
q2 = con.sql(f"""
    SELECT COUNT(*)                        AS n_rows,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_pages,
           MIN(report_date)                AS first_day,
           MAX(report_date)                AS last_day,
           COUNT(DISTINCT report_date)     AS n_days
    FROM {DAILY}
""").df()
q2

**Verification query 3: availability, filtered with `IS TRUE`.** The availability flags can be TRUE, FALSE or NULL. Filtering with `= FALSE` or `NOT ...` silently mishandles the NULL rows, so I count all three states and then show how many rows survive `gsc_data_available IS TRUE`, which is the filter I use for every feature.

In [ ]:
q3 = con.sql(f"""
    SELECT COUNT(*)                                                       AS rows_total,
           SUM(CASE WHEN gsc_data_available IS TRUE  THEN 1 ELSE 0 END)   AS gsc_true,
           SUM(CASE WHEN gsc_data_available IS FALSE THEN 1 ELSE 0 END)   AS gsc_false,
           SUM(CASE WHEN gsc_data_available IS NULL  THEN 1 ELSE 0 END)   AS gsc_null,
           SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END)   AS ga4_true
    FROM {DAILY}
""").df()
q3["share_surviving_gsc_is_true"] = q3["gsc_true"] / q3["rows_total"]
q3

### Five features, at most

I build a small feature frame for my lane from the same March partition, using **only** 1 to 15 March for the inputs and 16 to 31 March for the label. The first cell below builds it and the second shows my "knowable at the decision moment because..." line for each feature.

In [ ]:
frame = con.sql(f"""
    WITH base AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM {DAILY}
        WHERE gsc_data_available IS TRUE
    )
    SELECT client_hash_id, content_hash_id,
           SUM(CASE WHEN report_date <= DATE '{EARLY_END}' THEN gsc_impressions ELSE 0 END) AS imp_early,
           SUM(CASE WHEN report_date <= DATE '{EARLY_END}' THEN gsc_clicks      ELSE 0 END) AS clk_early,
           AVG(CASE WHEN report_date <= DATE '{EARLY_END}' AND gsc_impressions > 0
                    THEN gsc_avg_position END)                                               AS pos_early,
           COUNT(DISTINCT CASE WHEN report_date <= DATE '{EARLY_END}' AND gsc_impressions > 0
                    THEN report_date END)                                                    AS active_days_early,
           SUM(CASE WHEN report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-07'
                    THEN gsc_impressions ELSE 0 END)                                         AS imp_wk1,
           SUM(CASE WHEN report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-15'
                    THEN gsc_impressions ELSE 0 END)                                         AS imp_wk2,
           SUM(CASE WHEN report_date >= DATE '{LATE_START}' THEN gsc_impressions ELSE 0 END) AS imp_late
    FROM base
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '{EARLY_END}' THEN gsc_impressions ELSE 0 END) >= 100
""").df()

# five features, all built from 1-15 March only
frame["ctr_early_pct"] = frame["clk_early"] / frame["imp_early"] * 100
frame["early_trend"]   = ((frame["imp_wk2"] / 8) / (frame["imp_wk1"] / 7)).replace([np.inf, -np.inf], np.nan)

# label: built from the outcome window (16-31 March) versus the feature window
frame["declining"] = ((frame["imp_late"] / 16) < 0.8 * (frame["imp_early"] / 15)).astype(int)

FEATURES = ["imp_early", "ctr_early_pct", "pos_early", "active_days_early", "early_trend"]
print(f"{len(frame):,} pages kept (>= 100 impressions in 1-15 Mar) from {frame['client_hash_id'].nunique()} clients")
print(f"declining share (the label): {frame['declining'].mean():.1%}")
frame[FEATURES + ["declining"]].head(5)

In [ ]:
available_when = pd.DataFrame([
    ("imp_early",         "knowable at the decision moment because it only adds up impressions from 1-15 March, which are all in the past on 15 March."),
    ("ctr_early_pct",     "knowable at the decision moment because clicks and impressions from 1-15 March are already recorded by the end of 15 March."),
    ("pos_early",         "knowable at the decision moment because it averages the daily search position over 1-15 March only."),
    ("active_days_early", "knowable at the decision moment because it counts days with impressions from 1-15 March, a fact fixed by 15 March."),
    ("early_trend",       "knowable at the decision moment because it compares 8-15 March with 1-7 March, both finished by 15 March."),
], columns=["feature", "available when?"])
available_when.style.hide(axis="index").set_properties(**{"text-align": "left"})

### The trap: add one label-derived column on purpose

I score pages with a quick model (a gradient-boosted tree, split **by client** so a client is never in both train and test). First I score the five honest features. Then I add **one** column derived from the label, `late_over_early_ratio`, which is built from the outcome window. The score should jump toward perfect because the column is the answer in disguise. Then I delete it and keep the honest number.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

def quick_score(df, cols):
    """ROC-AUC on clients the model has never seen (0.5 = coin flip, 1.0 = perfect)."""
    y = df["declining"].values
    for seed in range(10):   # pick the first client split that has both classes on both sides
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=seed)
                      .split(df, y, groups=df["client_hash_id"]))
        if len(set(y[tr])) == 2 and len(set(y[te])) == 2:
            break
    model = HistGradientBoostingClassifier(max_iter=100, random_state=0).fit(df.iloc[tr][cols], y[tr])
    return roc_auc_score(y[te], model.predict_proba(df.iloc[te][cols])[:, 1])

honest = quick_score(frame, FEATURES)
print(f"Honest score, 5 features:                          AUC = {honest:.3f}")

# THE TRAP: a column computed from the outcome window (the label's own ingredient)
frame["late_over_early_ratio"] = (frame["imp_late"] / 16) / (frame["imp_early"] / 15)
leaky = quick_score(frame, FEATURES + ["late_over_early_ratio"])
print(f"With the leaky column added:                       AUC = {leaky:.3f}   <- looks amazing, teaches nothing")

# ...now delete it and keep the honest number
frame = frame.drop(columns=["late_over_early_ratio"])
assert "late_over_early_ratio" not in frame.columns
final = quick_score(frame, FEATURES)
print(f"Leaky column deleted, honest number kept:          AUC = {final:.3f}")

The score jumped when the label-derived column went in, and that jump is the warning sign: a feature that is only known *after* the outcome makes a model look perfect without learning anything. The honest number is the one printed after the column was deleted. It is the only one I would report, and I will describe it as an observed, directional result on one month of data.

## 4. Data limits

**The named limitation of my slice: it is a single month (March 2026) of an unbalanced panel.** One month cannot show seasonality, clients have different amounts of history, and a few large clients can supply many of the pages, so what I find describes **March 2026 for these clients**, not search behaviour in general. I also split the month at 15 March, so the outcome window (16 days) and the feature window (15 days) are short and uneven. The cell below measures how concentrated my frame is, so the limitation rests on a number and not on a guess.

In [ ]:
by_client = frame.groupby("client_hash_id").size().sort_values(ascending=False)
print(f"Pages in the modelling frame: {len(frame):,} from {by_client.size} clients")
print(f"Largest single client supplies {by_client.iloc[0] / len(frame):.1%} of the pages")
print(f"Top 3 clients supply {by_client.head(3).sum() / len(frame):.1%} of the pages")
print("Months covered by this slice: 1 (2026-03). No seasonality can be measured from it.")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.